#### Cash flow analytics + 7-day forecast

##### Goal: Calculate daily cash flow metrics and generate a 7-day revenue forecast using Facebook Prophet.

In [3]:
import os, json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import plotly.graph_objects as go
import plotly.express as px
from prophet import Prophet

import warnings
warnings.filterwarnings("ignore")

load_dotenv()
# engine = create_engine(os.getenv("DATABASE_URL"))

# with engine.connect() as conn:
#     count = conn.execute(text("SELECT COUNT(*) FROM transactions WHERE category IS NOT NULL")).scalar()
# print(f" Connected. {count} categorised transactions ready.")

True

#### Load Transactions for a Merchant

In [ ]:
def load_merchant_transactions(merchant_id: str, days: int = 90) -> pd.DataFrame:
    """
    Load the last N days of transactions for a moment.
    Returns a cleaned DataFrame ready for analysis.
    """
    
    query = """
        SELECT
            transaction_id,
            amount_ngn,
            type,
            category,
            channel,
            narration,
            timestamp::date AS date,
            timestamp
        FROM transactions
        WHERE merchant_id = :mid
            AND timestamp >= NOW() - INTERVAL ':days days'
        ORDER BY timestamp ASC
    """
    
    df = pd.read_sql(
        text("SELECT transaction_id, amount_ngn, type, category, channel, narration, "
             "timestamp::date AS date, timestamp FROM transactions "
             "WHERE merchant_id = :mid ORDER BY timestamp ASC"),
        engine, params={"mid": merchant_id}
    )
    
    if df.empty:
        print(f"No transactions found for {merchant_id}")
        return df
    
    df["date"] = pd.to_datetime(df["date"])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["amount_ngn"] = df["amount_ngn"].astype(float)
    
    print(f"Loaded {len(df)} transactions for {merchant_id}")
    print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
    return df

df = load_merchant_transaction("DEMO_001")
df.head()

#### Core Cash Flow Metrics

In [ ]:
def compute_cash_flow_metrics(df: pd.DataFrame) -> dict:
    """
    Compute all cash flow summary metrics for dashboard display.
    """
    if df.empty:
        return {}
    
    credits = df[df["type"] == "credit"]
    debits  = df[df["type"] == "debit"]
    
    total_revenue  = credits["amount_ngn"].sum()
    total_expenses = debits["amount_ngn"].sum()
    net_cash_flow  = total_revenue - total_expenses
    txn_count      = len(df)
    avg_txn_value  = credits["amount_ngn"].mean() if not credits.empty else 0
    
    # Daily revenue
    daily_rev = (
        credits.groupby("date")["amount_ngn"]
        .sum()
        .reset_index()
        .rename(columns={"amount_ngn": "revenue"})
    )
    
    # Week-over-week change
    today = df["date"].max()
    last7 = credits[credits["date"] >= today - timedelta(days=6)]["amount_ngn"].sum()
    prev7 = credits[
        (credits["date"] >= today - timedelta(days=13)) &
        (credits["date"] < today - timedelta(days=6))
    ]["amount_ngn"].sum()
    wow_change = ((last7 - prev7) / prev7 * 100) if prev7 > 0 else 0
    
    # Best revenue day
    best_day = daily_rev.loc[daily_rev["revenue"].idmax()] if not daily_rev.empty else None
    
    # Revenue by category
    cat_breakdown = (
        credits.groupby("category")["amount_ngn"]
        .sum()
        .sort_values(ascending=False)
        .to_dict()
    )
    
    # Revenue by channel
    channel_breakdown = (
        credits.groupby("category")["amount_ngn"]
        .sum()
        .sort_values(ascending=False)
        .to_dict()
    )
    
    return {
        "total_revenue_ngn":   round(total_revenue, 2),
        "total_expenses_ngn":  round(total_expenses, 2),
        "net_cash_flow_ngn":   round(net_cash_flow, 2),
        "transaction_count":   txn_count,
        "avg_transaction_ngn": round(avg_txn_value, 2),
        "wow_change_pct":      round(wow_change, 1),
        "last_7_days_revenue": round(last7, 2),
        "best_day":            str(best_day["date"]) if best_day is not None else None,
        "best_day_revenue":    round(best_day["revenue"], 2) if best_day is not None else 0,
        "daily_revenue":       daily_rev.to_dict(orient="records"),
        "category_breakdown":  cat_brakdown,
        "channel_breakdown":   channel_breakdown,
    }
    
metrics = compute_cash_flow-metrics(df)

print("Cash Flow Metrics")
print(f"   Total Revenue:   ₦{metrics['total_revenue_ngn']:>12,.2f}")
print(f"   Total Expenses:  ₦{metrics['total_expenses_ngn']:>12,.2f}")
print(f"   Net Cash Flow:   ₦{metrics['net_cash_flow_ngn']:>12,.2f}")
print(f"   Transactions:    {metrics['transaction_count']}")
print(f"   Avg Txn Value:   ₦{metrics['avg_transaction_ngn']:,.2f}")
print(f"   WoW Change:      {metrics['wow_change_pct']:+.1f}%")

#### 7-Day Revenue Forecast with Prophet

In [ ]:
def generate_forecast(df: pd.DataFrame, periods: int = 7) -> dict:
    """
    Generate a 7-day revenue forecast using Facebook Prophet.
    
    Prophet requires at least 2 data points (idealy 14+).
    Falls back to a simple moving average if data is too sparse.
    """
    
    credits = df[df["type"] == "credit"].copy()
    
    # Aggregate to daily revenue
    daily = (
        credits.groupby("date")["amount_ngn"]
        .sum()
        .reset_index()
        .rename(columns={"date": "ds", "amount_ngn": "y"})
    )
    daily["ds"] = pd.to_datetime(daily["ds"])
    
    # Fill missing days with 0
    date_range = pd.date_range(daily["ds"].min(), daily["ds"].max())
    daily = daily.set_index("ds").reindex(date_range, fill_value=0).reset_index()
    daily.columns = ["ds", "y"]
    
    if len(daily) < 2:
        # Fallback: simple average
        avg = credits["amount_ngn"].mean() if not credits.empty else 5000
        last_date = daily["ds"].max() if not daily.empty else datetime.now()
        forecast_dates = [last_date + timedelta(days=i+1) for i in range(periods)]
        return {
            "methods": "moving_average_fallback",
            "forecast": [
                {"date": str(d.date()), "predicted_revenue": round(avg, 2),
                 "lower_bound": round(avg * 0.7, 2), "upper_bound": round(avg * 1.3, 2)}
                for d in forecast_dates
            ],
            "total_predicted_7d": round(avg * periods, 2),
        }
        
    # Prophet model
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=False,
        changepoint_prior_scale=0.3,
        seasonality_prior_scale=10,
        interval_width=0.8,
    )
    
    # Nigerian public holidays as regressors (market closes / opens)
    nigeria_holidays = pd.DataFrame({
        "holiday": [
            "New Year", "Democracy Day", "Workers Day", "Eid al-Fitr", "Eid al-Adha", "Independence Day", "Christmas", "Boxing Day"
        ],
        "ds": pd.to_datetime([
            "2026-01-01", "2026-06-12", "2026-05-01", "2026-03-31",
            "2026-06-07", "2026-10-01", "2026-12-25", "2026-12-26"
        ]),
        'lower_window': 0,
        "upper_window": 1,
    })
    model = Prophet(
        holidays=nigerian_holidays,
        daily_seasonality=False,
        weekly_seasonality=True,
        changepoint_prior_scale=0.3,
        interval_width=0.8,
    )
    
    model.fit(daily)
    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)
    
    # Extract just the future forecast
    future_only = forecast.tail(periods)
    forecast_list = []
    for _, row in future_only.iterrows():
        forecast_list.append({
            "date": str(row["ds"].date()),
            "predicted_revenue": max(0, round(row["yhat"], 2)),
            "lower_bound":       max(0, round(row["yhat_lower"], 2)),
            "upper_bound":       max(0, round(row["yhat_upper"], 2)),
        })
        
    total_predicted = sum(f["predicted_revenue"] for f in forecast_list)
    
    return {
        "method": "prophet",
        "forecast": forecast_list,
        "total_predicted_7d": round(total_predicted, 2),
        "model_object": model,   # For plotting
        "forecast": forecast,    # For plotting
        "history_df": daily,
    }
    
    
forecast_result = generate_forecast(df)

print(f"Forecast generated using: {forecast_result['method']}")
print(f"   Total predicted 7-day revenue: ₦{forecast_result['total_predicted_7d']:,.2f}")
print()
print(" Daily Forecast:")
for day in forecast_result["forecast"]:
    print(f"   {day['date']}  ₦{day['predicted_revenue']:>10,.2f}  "
          f"(₦{day['lower_bound']:,.0f} – ₦{day['upper_bound']:,.0f})")

#### Visualize: Cash Flow Trend Chart

In [ ]:
# Daily revenue + expenses chart
credits = df[df["type"] == "credit"]
debits  = df[df["type"] == "debit"]

daily_credits = credits.groupby("date")["amount_ngn"].sum().reset_index()
daily_debits  = debits.groupby("date")["amount_ngn"].sum().reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=daily_credits["date"], y=daily_credits["amount_ngn"],
    name="Revenue (Credits)", marker_color="#22c55e"
))
fig.add_trace(go.Bar(
    x=daily_debits["date"], y=daily_debits["amount_ngn"],
    name="Expenses (Debits)", marker_color="#ef4444"
))
fig.update_layout(
    title="Daily Cash Flow — Mama Titi Kitchen (DEMO_001)",
    xaxis_title="Date", yaxis_title="Amount (₦)",
    barmode="group", template="plotly_white", height=400
)
fig.show()

#### Visualize: 7-Day Forecast Chart

In [ ]:
if forecast_result["methods"] == "prophet":
    forecast_df = forecast_result["forecast_df"]
    history_df  = forecast_result["history_df"]
    last_date   = history_df["ds"].max()
    
    hist_plot   = forecast_df[forecast_df["ds"] <= last_date]
    future_plot = forecast_df[forecast_df["ds"] >  last_date]
    
    fig = go.Figure()
    # Actual history
    fig.add_trace(go.Scatter(
        x=history_df["ds"], y=history_df["y"],
        name="Actual Revenue", mode="markers+lines",
        line=dict(color="#3b82f6", width=2)
    ))
    # Forecast
    fig.add_trace(go.Scatter(
        x=future_plot["ds"], y=future_plot["yhat"],
        name="Forecast", mode="lines",
        line=dict(color="#f59e0b", width=2, dash="dash")
    ))
    # Confidence band
    fig.add_trace(go.Scatter(
        x=pd.concat([future_plot["ds"], future_plot["ds"][::-1]]),
        y=pd.concat([future_plot["yhat_upper"], future_plot["yhat_lower"][::-1]]),
        fill="toself", fillcolor="rgba(245,158,11,0.15)",
        line=dict(color="rgba(255,255,255,0)"), name="80% Confidence"
    ))
    fig.update_layout(
        title="7-Day Revenue Forecast (Prophet)",
        xaxis_title="Date", yaxis_title="Revenue (₦)",
        template="plotly_white", height=400
    )
    fig.show()
else:
    forecast_items = forecast_result["forecast"]
    dates = [f["date"] for f in forecast_items]
    revs  = [f["predicted_revenue"] for f in forecast_items]
    fig = px.bar(x=dates, y=revs, title="7-Day Revenue Forecast (Moving Avg)",
                 labels={"x": "Date", "y": "Revenue (₦)"})
    fig.show()

#### Category Spend Breakdown

In [ ]:
cat_data = pd.DataFrame(
    list(metrics["category_breakdown"].items()),
    columns=["category", "amount_ngn"]
).sort_values("amount_ngn", ascending=True)

fig = px.bar(
    cat_data, x="amount_ngn", y="category", orientation="h",
    title="Revenue by Category", labels={"amount_ngn": "Revenue (₦)", "category": "Category"},
    color="amount_ngn", color_continuous_scale="Greens"
)
fig.update_layout(template="plotly_white", height=350)
fig.show()

#### Write src/analytics/cashflow.py

In [ ]:
os.makedirs("src/analytics", exist_ok=True)

cashflow_py = '''
import os
import warnings
import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
from prophet import Prophet
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")


def load_merchant_transactions(merchant_id: str, db_engine=None) -> pd.DataFrame:
    if db_engine is None:
        db_engine = create_engine(os.getenv("DATABASE_URL"))
    df = pd.read_sql(
        text("SELECT transaction_id, amount_ngn, type, category, channel, narration, "
             "timestamp::date AS date, timestamp FROM transactions "
             "WHERE merchant_id = :mid ORDER BY timestamp ASC"),
        db_engine, params={"mid": merchant_id}
    )
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"])
        df["amount_ngn"] = df["amount_ngn"].astype(float)
    return df


def compute_cash_flow_metrics(df: pd.DataFrame) -> dict:
    if df.empty:
        return {}
    credits = df[df["type"] == "credit"]
    debits  = df[df["type"] == "debit"]
    total_revenue  = credits["amount_ngn"].sum()
    total_expenses = debits["amount_ngn"].sum()
    net_cash_flow  = total_revenue - total_expenses
    avg_txn_value  = credits["amount_ngn"].mean() if not credits.empty else 0
    daily_rev = credits.groupby("date")["amount_ngn"].sum().reset_index().rename(columns={"amount_ngn": "revenue"})
    today = df["date"].max()
    last7 = credits[credits["date"] >= today - timedelta(days=6)]["amount_ngn"].sum()
    prev7 = credits[(credits["date"] >= today - timedelta(days=13)) & (credits["date"] < today - timedelta(days=6))]["amount_ngn"].sum()
    wow_change = ((last7 - prev7) / prev7 * 100) if prev7 > 0 else 0
    best_day = daily_rev.loc[daily_rev["revenue"].idxmax()] if not daily_rev.empty else None
    return {
        "total_revenue_ngn":   round(total_revenue, 2),
        "total_expenses_ngn":  round(total_expenses, 2),
        "net_cash_flow_ngn":   round(net_cash_flow, 2),
        "transaction_count":   len(df),
        "avg_transaction_ngn": round(avg_txn_value, 2),
        "wow_change_pct":      round(wow_change, 1),
        "last_7_days_revenue": round(last7, 2),
        "best_day":            str(best_day["date"]) if best_day is not None else None,
        "best_day_revenue":    round(best_day["revenue"], 2) if best_day is not None else 0,
        "daily_revenue":       daily_rev.to_dict(orient="records"),
        "category_breakdown":  credits.groupby("category")["amount_ngn"].sum().sort_values(ascending=False).to_dict(),
        "channel_breakdown":   credits.groupby("channel")["amount_ngn"].sum().sort_values(ascending=False).to_dict(),
    }


def generate_forecast(df: pd.DataFrame, periods: int = 7) -> dict:
    credits = df[df["type"] == "credit"].copy()
    daily = credits.groupby("date")["amount_ngn"].sum().reset_index().rename(columns={"date": "ds", "amount_ngn": "y"})
    daily["ds"] = pd.to_datetime(daily["ds"])
    if len(daily) < 2:
        avg = credits["amount_ngn"].mean() if not credits.empty else 5000
        last_date = daily["ds"].max() if not daily.empty else datetime.now()
        return {
            "method": "moving_average_fallback",
            "forecast": [{"date": str((last_date + timedelta(days=i+1)).date()),
                          "predicted_revenue": round(avg, 2),
                          "lower_bound": round(avg * 0.7, 2),
                          "upper_bound": round(avg * 1.3, 2)} for i in range(periods)],
            "total_predicted_7d": round(avg * periods, 2),
        }
    date_range = pd.date_range(daily["ds"].min(), daily["ds"].max())
    daily = daily.set_index("ds").reindex(date_range, fill_value=0).reset_index()
    daily.columns = ["ds", "y"]
    nigerian_holidays = pd.DataFrame({
        "holiday": ["New Year","Democracy Day","Workers Day","Eid al-Fitr","Eid al-Adha","Independence Day","Christmas","Boxing Day"],
        "ds": pd.to_datetime(["2026-01-01","2026-06-12","2026-05-01","2026-03-31","2026-06-07","2026-10-01","2026-12-25","2026-12-26"]),
        "lower_window": 0, "upper_window": 1,
    })
    model = Prophet(holidays=nigerian_holidays, weekly_seasonality=True, daily_seasonality=False, interval_width=0.8)
    model.fit(daily)
    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)
    future_only = forecast.tail(periods)
    forecast_list = [{"date": str(r["ds"].date()), "predicted_revenue": max(0, round(r["yhat"], 2)),
                      "lower_bound": max(0, round(r["yhat_lower"], 2)), "upper_bound": max(0, round(r["yhat_upper"], 2))}
                     for _, r in future_only.iterrows()]
    return {
        "method": "prophet",
        "forecast": forecast_list,
        "total_predicted_7d": round(sum(f["predicted_revenue"] for f in forecast_list), 2),
    }


def get_analytics(merchant_id: str, db_engine=None) -> dict:
    """Main entry point — returns combined metrics + forecast."""
    df = load_merchant_transactions(merchant_id, db_engine)
    metrics = compute_cash_flow_metrics(df)
    forecast = generate_forecast(df)
    return {**metrics, "forecast_7day": forecast}
'''

with open("src/analytics/cashflow.py", "w") as f:
    f.write(cashflow_py.strip())

print(" src/analytics/cashflow.py written.")

#### Add /analytics Endpoint to FastAPI

In [ ]:
analytics_route = '''

# ── Analytics endpoint (add to src/api/main.py) ───────────────────────
@app.get("/merchant/{merchant_id}/analytics")
async def get_analytics(merchant_id: str):
    from src.analytics.cashflow import get_analytics
    result = get_analytics(merchant_id, ENGINE)
    if not result:
        raise HTTPException(status_code=404, detail=f"No data for merchant {merchant_id}")
    return result
'''

with open("src/api/analytics_route.py", "w") as f:
    f.write(analytics_route.strip())

print("analytics_route.py written — paste contents into src/api/main.py")

### Done Test

In [ ]:
# ── THE DAY 3 DONE TEST ─────────────────────────────────────────────────
result = {**metrics, "forecast_7day": forecast_result}

checks = {
    "daily_revenue populated":       len(result.get("daily_revenue", [])) > 0,
    "total_revenue_ngn > 0":         result.get("total_revenue_ngn", 0) > 0,
    "forecast_7day present":         "forecast_7day" in result,
    "7 forecast days":               len(result["forecast_7day"].get("forecast", [])) == 7,
    "forecast has predicted_revenue": all("predicted_revenue" in f for f in result["forecast_7day"]["forecast"]),
    "category_breakdown present":    len(result.get("category_breakdown", {})) > 0,
    "channel_breakdown present":     len(result.get("channel_breakdown", {})) > 0,
}

all_passed = all(checks.values())
for check, passed in checks.items():
    print(f"  {if passed else} {check}")

print()
if all_passed:
    print("DAY 3 DONE TEST PASSED")
    print(f"   Revenue: ₦{result['total_revenue_ngn']:,.2f}  |  Net: ₦{result['net_cash_flow_ngn']:,.2f}")
    print(f"   7-day forecast total: ₦{result['forecast_7day']['total_predicted_7d']:,.2f}")
else:
    print("Some checks failed — review cells above.")

print()
print("Day 3 complete. Move to Day 4: Credit Readiness Score.")